# TUGAS AKHIR 03 (TA-03): PEMODELAN DAN SIMULASI
**Topik:** Data-Driven Monte Carlo Simulation  
**Studi Kasus:** Estimasi Kerugian Finansial Akibat Pembatalan Pemesanan Hotel  
**Nama:** Soni Moch Leviansyah  
**NIM:** 301240030  

---
## 1. Import Library & Pengaturan Seed NIM
Langkah awal adalah mengimpor library komputasi numerik, manipulasi data, dan visualisasi, serta mengunci nilai acak (*random seed*) menggunakan NIM mahasiswa agar hasil simulasi konsisten dan teruji.

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Mengunci seed dengan NIM
NIM = 301240030
np.random.seed(NIM)

print(f"Random seed berhasil diatur dengan NIM: {NIM}")

Random seed berhasil diatur dengan NIM: 301240030


## 2. Load Dataset & Data Cleaning
Dataset yang digunakan adalah `hotel_bookings.csv` dari Kaggle. Pada tahap ini dilakukan pembersihan data sederhana dengan menyaring tarif harian kamar (*Average Daily Rate* / `adr`) yang bernilai valid (> 0).

In [7]:
# Load dataset
df = pd.read_csv('hotel_bookings.csv')

# Data cleaning: Ambil data ADR yang bernilai lebih besar dari 0
df_clean = df[df['adr'] > 0].copy()

# Tampilkan 5 baris pertama data yang sudah dibersihkan
df_clean[['hotel', 'is_canceled', 'adr', 'country']].head()

,hotel,is_canceled,adr,country
2,Resort Hotel,0,75.0,GBR
3,Resort Hotel,0,75.0,GBR
4,Resort Hotel,0,98.0,GBR
5,Resort Hotel,0,98.0,GBR
6,Resort Hotel,0,107.0,PRT


## 3. Ekstraksi Parameter dari Data Historis
Menghitung nilai statistik deskriptif historis dari kolom `adr` (Rata-rata $\mu$ dan Standar Deviasi $\sigma$) serta rasio pembatalan historis (`is_canceled`). Parameter ini akan digunakan sebagai dasar pembangkitan angka acak dalam simulasi Monte Carlo.

In [8]:
# Menghitung parameter statistik
mean_adr = df_clean['adr'].mean()
std_adr = df_clean['adr'].std()
cancellation_rate = df_clean['is_canceled'].mean()

print("=" * 50)
print("          PARAMETER STATISTIK HISTORIS          ")
print("=" * 50)
print(f"Rata-rata Harga Kamar (Mean / μ) : ${mean_adr:.2f}")
print(f"Standar Deviasi (Std Dev / σ) : ${std_adr:.2f}")
print(f"Rasio Pembatalan (Cancel Rate): {cancellation_rate * 100:.2f}%")
print("=" * 50)

# Visualisasi histogram distribusi data asli
plt.figure(figsize=(8, 4))
sns.histplot(df_clean['adr'], bins=50, kde=True, color='skyblue')
plt.title("Distribusi Harga Kamar (ADR) Historis")
plt.xlabel("Average Daily Rate ($)")
plt.ylabel("Frekuensi")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

          PARAMETER STATISTIK HISTORIS          
Rata-rata Harga Kamar (Mean / μ) : $103.53
Standar Deviasi (Std Dev / σ) : $49.20
Rasio Pembatalan (Cancel Rate): 37.48%


C:\Users\USER\AppData\Local\Temp\ipykernel_24472\2423898280.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Membangun Simulasi Monte Carlo
Melakukan simulasi sebanyak 5.000 iterasi. Pada setiap iterasi, disimulasikan 100 pemesanan kamar baru (*bookings*). Harga kamar dibangkitkan secara acak dari distribusi normal berdasarkan parameter $\mu$ dan $\sigma$, sedangkan status pembatalan ditentukan melalui probabilitas rasio pembatalan historis.

In [9]:
num_simulations = 5000  # Minimal 5.000 iterasi
num_bookings = 100      # Proyeksi 100 booking baru

simulated_losses = []

for _ in range(num_simulations):
    # 1. Pembangkitan angka acak harga kamar dari Distribusi Normal (μ, σ)
    prices = np.random.normal(mean_adr, std_adr, num_bookings)
    prices = np.maximum(prices, 0) # Menghindari harga negatif
    
    # 2. Penentuan kejadian pembatalan (Bernoulli Process)
    is_canceled = np.random.rand(num_bookings) < cancellation_rate
    
    # 3. Hitung total kerugian akibat pembatalan pada iterasi ini
    total_loss = np.sum(prices[is_canceled])
    simulated_losses.append(total_loss)

simulated_losses = np.array(simulated_losses)
print(f"Simulasi Monte Carlo sebanyak {num_simulations:,} iterasi selesai dijalankan.")

Simulasi Monte Carlo sebanyak 5,000 iterasi selesai dijalankan.


## 5. Analisis Risiko & Visualisasi Hasil Simulasi
Menghitung probabilitas terjadinya risiko kerugian yang melampaui batas ambang batas (*threshold*) sebesar $5.500, serta menampilkan kurva distribusi hasil simulasi Monte Carlo.

In [10]:
# Batas ambang batas risiko kerugian
threshold_loss = 5500.0

# Menghitung skenario yang melebihi threshold
exceeded_count = np.sum(simulated_losses > threshold_loss)
risk_probability = (exceeded_count / num_simulations) * 100

print("=" * 50)
print("       HASIL ANALISIS RISIKO MONTE CARLO        ")
print("=" * 50)
print(f"Total Iterasi Skenario               : {num_simulations:,}")
print(f"Jumlah Skenario > Threshold ($5,500) : {exceeded_count:,}")
print(f"Probabilitas Kerugian > $5,500        : {risk_probability:.2f}%")
print("=" * 50)

# Visualisasi Hasil Simulasi
plt.figure(figsize=(10, 5))
sns.histplot(simulated_losses, kde=True, color='crimson', bins=40)
plt.axvline(threshold_loss, color='black', linestyle='--', linewidth=2, 
            label=f'Threshold Risiko (${threshold_loss:,.0f})')
plt.title("Visualisasi Risiko Kerugian Pembatalan Hotel (Simulasi Monte Carlo)")
plt.xlabel("Total Kerugian Finansial ($)")
plt.ylabel("Frekuensi Skenario")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

       HASIL ANALISIS RISIKO MONTE CARLO        
Total Iterasi Skenario               : 5,000
Jumlah Skenario > Threshold ($5,500) : 20
Probabilitas Kerugian > $5,500        : 0.40%


C:\Users\USER\AppData\Local\Temp\ipykernel_24472\4100743434.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Kesimpulan Bisnis
Berdasarkan hasil simulasi Monte Carlo sebanyak **5.000 iterasi** menggunakan parameter dari dataset historis Kaggle `hotel_bookings.csv`:
1. Rata-rata estimasi kerugian akibat pembatalan pemesanan hotel berkisar pada rentang **$3.800 – $4.000**.
2. Probabilitas terjadinya kerugian finansial yang melebihi ambang batas toleransi anggaran sebesar **$5.500** adalah sangat kecil, yaitu sebesar **0.43%**.
3. Hasil ini menunjukkan bahwa tingkat risiko finansial dari pembatalan 100 kamar baru tergolong aman dan masih dalam batas toleransi manajemen hotel.